# Problem 2

In [17]:
import numpy as np

In [ ]:
x_k = np.array([0, 2, 4, 6, 8, 10], dtype=float)
E_k = np.array([210, 205, 190, 185, 200, 
                220]) * 1e9

In [11]:
def I_func(x):
    return 0.005 + 0.001 * np.sin(x)

def newton_coeffs(xs, ys):
    n = len(xs)
    coef = ys.astype(float).copy()
    for j in range(1, n):
        coef[j:n] = (coef[j:n] - coef[j - 1:n - 1]) / (xs[j:n] - xs[0:n - j])
    return coef

def newton_eval(xs, coef, x):
    n = len(xs)
    p = coef[-1]
    for k in range(n - 2, -1, -1):
        p = p * (x - xs[k]) + coef[k]
    return p

def E_interp(x):
    x = np.atleast_1d(np.asarray(x, dtype=float))
    out = np.zeros_like(x)
    n = len(x_k)
    for i, xi in enumerate(x):
        k = min(max(np.searchsorted(x_k, xi) - 1, 0), n - 2)
        lo = max(k - 1, 0)
        hi = min(lo + 4, n)
        lo = max(hi - 4, 0)
        xs, ys = x_k[lo:hi], E_k[lo:hi]
        coef = newton_coeffs(xs, ys)
        out[i] = newton_eval(xs, coef, xi)
    return out if out.size > 1 else out[0]

Local cubic (4-point) interpolation is used on each panel instead of a single global 5th-degree
polynomial across all 6 points, because a high-degree global fit through unevenly-behaved data is
prone to Runge-type oscillation between nodes — the local piecewise fit stays close to the actual
measured trend and doesn't blow up near the edges.

### Task 2 — dual-method quadrature

In [12]:
def integrand(x, P):
    return (P * (10 - x)) ** 2 / (2 * E_interp(x) * I_func(x))

def trapezoidal(g, a, b, N):
    x = np.linspace(a, b, N + 1)
    y = g(x)
    h = (b - a) / N
    return h * (0.5 * y[0] + 0.5 * y[-1] + np.sum(y[1:-1]))

def gauss_legendre_3pt():
    nodes = np.array([-np.sqrt(3 / 5), 0.0, np.sqrt(3 / 5)])
    weights = np.array([5 / 9, 8 / 9, 5 / 9])
    return nodes, weights

def calc_energy(P, method="A"):
    if method == "A":
        return trapezoidal(lambda x: integrand(x, P), 0, 10, 100)
    nodes, weights = gauss_legendre_3pt()
    total = 0.0
    for k in range(len(x_k) - 1):
        a, b = x_k[k], x_k[k + 1]
        xm = (b - a) / 2 * nodes + (a + b) / 2
        total += (b - a) / 2 * np.sum(weights * integrand(xm, P))
    return total

print("Method A:", calc_energy(50000, "A"))
print("Method B:", calc_energy(50000, "B"))

Method A: 400.36286364300344
Method B: 400.3227629732594


### Task 3 — Non-linear Newton-Raphson Synthesis

In [13]:
def solve_load(target=1.5e6, P0=50000.0, tol=1e-6, h=1.0, max_iter=100):
    P = P0
    for it in range(max_iter):
        g = calc_energy(P, "B") - target
        dg = (calc_energy(P + h, "B") - calc_energy(P - h, "B")) / (2 * h)
        P_new = P - g / dg
        if abs(P_new - P) < tol:
            return P_new, it + 1
        P = P_new
    return P, max_iter

P_star, iters = solve_load()
print("P* =", P_star, "N, iterations:", iters)
print("check Eb(P*) =", calc_energy(P_star, "B"))

P* = 3060627.605892556 N, iterations: 11
check Eb(P*) = 1500000.0000000002


## Problem 3 

In [14]:
def simpson_weights(N):
    w = np.ones(N + 1)
    w[1:-1:2] = 4
    w[2:-1:2] = 2
    return w

def q(x, y):
    return np.exp(-(x ** 2 + y ** 2)) * np.cos(x + y)

### Method 1 — polar

In [15]:
def method1_polar(Nr=100, Ntheta=100, R=2.0):
    theta = np.linspace(0, np.pi, Ntheta + 1)
    htheta = np.pi / Ntheta
    wtheta = simpson_weights(Ntheta)
    r = np.linspace(0, R, Nr + 1)
    hr = R / Nr
    wr = simpson_weights(Nr)

    inner = np.zeros(Ntheta + 1)
    for i, th in enumerate(theta):
        x = r * np.cos(th)
        y = r * np.sin(th)
        g = q(x, y) * r
        inner[i] = (hr / 3) * np.sum(wr * g)
    return (htheta / 3) * np.sum(wtheta * inner)

Q1 = method1_polar()
print("Q (polar):", Q1)

Q (polar): 0.9610458549898161


### Method 2 — Cartesian, variable y limit

In [16]:
def method2_cartesian(Nx=100, Ny=100):
    a, b = -2.0, 2.0
    x = np.linspace(a, b, Nx + 1)
    hx = (b - a) / Nx
    wx = simpson_weights(Nx)

    inner = np.zeros(Nx + 1)
    for i, xi in enumerate(x):
        ymax = np.sqrt(max(4 - xi ** 2, 0.0))
        if ymax == 0.0:
            continue
        y = np.linspace(0, ymax, Ny + 1)
        hy = ymax / Ny
        wy = simpson_weights(Ny)
        g = q(xi, y)
        inner[i] = (hy / 3) * np.sum(wy * g)
    return (hx / 3) * np.sum(wx * inner)

Q2 = method2_cartesian()
print("Q (cartesian):", Q2)
print("difference:", abs(Q1 - Q2))

Q (cartesian): 0.9610656567994064
difference: 1.980180959026967e-05


Both give close results, but the Cartesian version discretizes a curved boundary with a
rectangular grid — near x = ±2 the panels get squeezed against y_max(x) which goes to zero, and the
per-column Simpson step size hy changes with x, adding extra discretization error at the edge of
the semicircle. Polar coordinates map the domain to an exact rectangle [0,2]x[0,pi], so the same
uniform Nr, Ntheta grid covers Omega exactly with no boundary approximation.